# Bouncing ball - analysis

A sphere is dropped on the ground and bounces with the impulse model, restitution coefficient
e < 1: each bounce removes a fraction 1 − e² of the energy. The scene, restitution, simulated time and dt range are defined in `main.cpp`. Its results:

- `results/benchmark.csv`: one row per (solver, dt), with the metrics below;
- `results/energy_drift.csv`: E(t) for a few dt values.

Run `./scripts/physics_benchmark.sh --scenarios Bouncing` to produce them and execute this notebook.

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, "..")
from analysis_utils import RESULTS, STYLE, expected_orders, export_pdf, plot_solvers, save, shade_from, time_series

df = pd.read_csv(RESULTS / "benchmark.csv")
ede = pd.read_csv(RESULTS / "energy_drift.csv")
print(f"benchmark.csv: {len(df)} rows, energy_drift.csv: {len(ede)} rows")

## Trusted dt range

At large dt a run can miss or add bounces (for instance when the sphere moves past the contact zone
within a single step). Each solver's bounce count at its finest dt is taken as the reference; from
the first dt where any solver differs from it, the results are greyed out in the plots and left out
of the cost plot.

In [ ]:
# Reference bounce count: each solver's count at its finest dt. From the first dt where any solver
# differs from it, the runs miss or add bounces and their errors are not comparable.
ref_count = df.loc[df.groupby("solver")["dt"].idxmin()].set_index("solver")["bounce_count"]
wrong = df[df["bounce_count"] != df["solver"].map(ref_count)]
dt_wrong = wrong["dt"].min() if not wrong.empty else None
valid = df if dt_wrong is None else df[df["dt"] < dt_wrong]
print("Bounce count at the finest dt:", ref_count.to_dict())
print("First dt with a different bounce count:", dt_wrong)

## Bounce count vs dt

**How it is made.** Number of bounces detected during the run: steps where the vertical velocity
turns from negative to positive. Dashed line: the reference count (finest dt).

**What to expect.** The same count at every dt, until dt becomes too large to resolve the contact.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
plot_solvers(ax, df, "dt", "bounce_count", logy=False)
for count in sorted(set(ref_count)):
    ax.axhline(count, color="grey", ls="--", lw=1, label="count at the finest dt")
shade_from(ax, dt_wrong, "different bounce count")
ax.set_xlabel("dt  (s)")
ax.set_ylabel("Bounces detected")
ax.set_title("Bounce count vs dt")
ax.legend(fontsize=8)
save(fig, "plot_bounce_count")

## Peak height error vs dt

**How it is made.** After each bounce, the highest sampled height before the velocity turns
negative is compared with the analytical apex `z_peak(n) = r + (z₀ − r)·e^(2n)`. The plotted value is
the largest relative error over all bounces. Dotted lines: slope 1.

**What to expect.** Two sources of error:
- free flight: Euler is first order, Verlet and RK4 are exact for constant gravity;
- contact: the bounce is applied at the step where contact is detected, not at the exact contact
  time, and the apex is the highest *sampled* point. This error depends on dt for every solver and
  is first order in general.

So all three solvers are expected to converge with **slope 1**, Euler with a larger error.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_solvers(ax, df, "dt", "max_height_error")
expected_orders(ax, valid, "dt", "max_height_error", {"Euler": 1, "Verlet": 1, "RK4": 1})
shade_from(ax, dt_wrong, "different bounce count")
ax.set_xlabel("dt  (s)")
ax.set_ylabel("max relative peak height error")
ax.set_title("Peak height error vs dt")
ax.legend(ncol=2, fontsize=8)
save(fig, "plot_convergence")

## Free-flight energy drift vs dt

**How it is made.** Between bounces, while the sphere is clearly above the ground (threshold in
`main.cpp`), the energy is compared with its value just after the last bounce:
`max |E − E_ref| / E_ref`. This isolates the integrator from the contact handling.

**What to expect.** Euler loses energy steadily in flight: **slope 1**. Verlet and RK4 are exact for
constant gravity: only round-off, near the double precision ε and growing slowly with the number
of steps (∝ 1/dt).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_solvers(ax, df, "dt", "max_flight_energy_drift")
expected_orders(ax, valid, "dt", "max_flight_energy_drift", {"Euler": 1})
ax.axhline(np.finfo(float).eps, color="grey", ls="--", lw=1, label="double precision ε")
shade_from(ax, dt_wrong, "different bounce count")
ax.set_xlabel("dt  (s)")
ax.set_ylabel("max |E − E_ref| / E_ref  (in flight)")
ax.set_title("Free-flight energy drift vs dt")
ax.legend(ncol=2, fontsize=8)
save(fig, "plot_flight_energy_drift")

## Cost vs precision

**How it is made.** Same error as the convergence plot, against the cost of the run: the number of
force evaluations, `ops_total` = steps × evaluations per step (1 for Euler and Verlet, 4 for RK4, as
counted in `main.cpp`).

**What to expect.** At a given cost (vertical line), the lowest curve is the most accurate solver
for that budget. A higher order pays off only if its error falls faster than its extra cost.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_solvers(ax, valid, "ops_total", "max_height_error")
ax.set_xlabel("Force evaluations over the run  (cost)")
ax.set_ylabel("max relative peak height error")
ax.set_title("Cost vs precision")
ax.legend(fontsize=8)
save(fig, "plot_cost_vs_precision")

## CPU time vs dt

**How it is made.** Wall time of the simulation loop, in µs, for each run. Runs shorter than
the timer resolution (1 µs) read 0 and are not shown. The dotted lines have slope −1.

**What to expect.** The time is proportional to the number of steps, so to 1/dt (slope −1). RK4
evaluates the forces 4 times per step, but the rest of the per-step work does not scale with it, so
RK4 should cost less than 4× Euler.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_solvers(ax, df, "dt", "cpu_us")
expected_orders(ax, df, "dt", "cpu_us", {"Euler": -1, "Verlet": -1, "RK4": -1})
ax.set_xlabel("dt  (s)")
ax.set_ylabel("CPU time  (µs)")
ax.set_title("CPU time vs dt")
ax.legend(ncol=2, fontsize=8)
save(fig, "plot_cpu_time")

## Energy over time

**How it is made.** `|E(t) − E₀| / E₀` along the run, sub-sampled, for the dt values recorded by
`main.cpp`.

**What to expect.** A step at each bounce, as the energy falls to e^(2n)·E₀ after bounce n (the same for every solver); in between, Euler drifts and Verlet and RK4 stay flat.
On a log scale of an absolute value, sharp dips are points where `E(t) − E₀` changes sign.

In [ ]:
fig = time_series(ede)
fig.suptitle("Energy drift over time", y=1.0)
save(fig, "plot_energy_time_series")

## PDF report

In [ ]:
export_pdf()